In [10]:
pip install -q transformers einops accelerate langchain bitsandbytes ipywidgets langchain-huggingface transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
!nvidia-smi

Tue Feb 24 06:10:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 577.05                 Driver Version: 577.05         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 ...  WDDM  |   00000000:02:00.0 Off |                  N/A |
| N/A   35C    P0             11W /   95W |       0MiB /  12227MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, pipeline
import torch

model_id = "tiiuae/falcon-7b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen_pipe = pipeline(
    task="text-generation",
    model=model_id,
    tokenizer=tokenizer,
    device_map="auto",
    dtype=torch.float16,
    max_new_tokens=200,
    do_sample=True,
    top_k=10,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
Device set to use cpu


In [13]:
llm = HuggingFacePipeline(pipeline = gen_pipe, model_kwargs = {'temperature':0})

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = """
Write a short nursery rhyme for children (8 lines max) explaining Artificial Intelligence.
- Must rhyme (AABB or ABAB)
- Simple words (kid-friendly)
- Mention: learning, helping, and computers
Question: {question}
Answer:
"""

prompt = PromptTemplate.from_template(template)

chain = prompt | llm | StrOutputParser()

question = "Explain what is Artificial Intelligence as Nursery Rhymes"

response = chain.invoke({"question": question})

print(response)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.



Write a short nursery rhyme for children (8 lines max) explaining Artificial Intelligence.
- Must rhyme (AABB or ABAB)
- Simple words (kid-friendly)
- Mention: learning, helping, and computers
Question: Explain what is Artificial Intelligence as Nursery Rhymes
Answer:
Computers help us learn; 
They can do amazing things. 
They think and work like humans; 
But they do it much quicker! 
They can help you find your way; 
Like a guide that's always there. 
They make life so much easier; 
Thanks to Artificial Intelligence!
